# Ch.3  機械学習で品種を分類する

受講生用ノートブック

| 項目 | 内容 |
|------|------|
| 使うデータ | ワインデータ 178件・13特徴量・3品種 |
| 演習時間 | 35分（問3まで完了で合格） |
| ゴール | RandomForest でモデルを作り、正解率と特徴量重要度を確認する |

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇をしたい / 〇〇を確認したい
【使うライブラリ】scikit-learn / pandas
【データの形】df という DataFrame、178行×14列（数値13列 + 品種名列）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

💡 Ch.3 の一言アドバイス：
「使うクラス名（RandomForestClassifier など）と、X・y の形を具体的に伝えると
 精度の高いコードが返ってきます。」

「モデルを作りたい」ではなく、
「X_train と y_train を使って RandomForestClassifier を学習させたい、どう書くか」のように、
**変数名・クラス名・やりたいこと** をセットで伝えましょう。

---
## 準備  ライブラリとデータを読み込む

⛔ 注意 最初にこのセルを実行してください。

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import japanize_matplotlib

wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = ["wine_" + str(t) for t in wine.target]
print(f"行数: {len(df)}  列数: {len(df.columns)}")

---
## なぜ「機械学習」を使うのか

Ch.1・Ch.2 で「alcohol と proline で品種が分かれそう」という仮説を立てました。

人間が「alcohol >= 13.5 かつ proline >= 700 なら wine_0」と手動でルールを作ることも可能ですが：

| 方法 | 問題点 |
|------|--------|
| 人間のルール | 変数が 13 個あると組み合わせが膨大 |
| 人間のルール | 新しいデータに対応できない |
| 機械学習（RF） | 13 変数すべてを使って最適なルールを自動生成 |

今回使う RandomForest：
- 決定木（木の形のルール）をたくさん作って多数決する
- 「どの変数がどれだけ判断に使われたか（特徴量重要度）」がわかる

---
## STEP 1  モデルへの「入力」と「答え」を分ける

機械学習の学習に必要なのは：
- X（特徴量）: モデルが「見る」数値データ → 13 列の数値
- y（ターゲット）: モデルが「当てる」答え → 品種（0/1/2）

---

Q1-1：特徴量（X）とターゲット（y）に分けてみましょう

- X：品種名を除いた数値列 13 個
- y：品種名列

💡 ヒント：「特定の列を除いた DataFrame と、その列だけを別変数に分ける方法」を Copilot に聞いてみましょう。

In [ ]:
# 品種名以外の数値列を特徴量（X）、品種名列をターゲット（y）として分ける

---
Q1-1（追加）  X と y の形を確認しましょう

X と y が正しく分割できているか、行数・列数を数値で確認します。
「X は 178行×13列、y は 178件」になっていれば成功です。

💡 ヒント：「DataFrame と Series の行数・列数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# X の行数・列数と y の件数を確認して、正しく分割できているか検証する

---
Q1-2：データを「学習用」と「テスト用」に分けましょう

モデルは学習に使ったデータでは高い精度が出ます。
未知のデータ（テスト用）でも正しく予測できるかを確認するため、事前に分割します。

学習用：テスト用 = 8：2 に分割してください。

💡 ヒント：「DataFrame を学習用とテスト用に 80:20 で分割する方法」を Copilot に聞いてみましょう。

In [ ]:
# データを学習用（80%）とテスト用（20%）に分割し、モデルの評価を公正に行えるように準備する

---
Q1-2（追加）  分割後のデータ件数とクラス分布を確認しましょう

80:20 に分割すると、実際に何件が訓練用・何件がテスト用になるかを確認します。
また、訓練データに3品種がバランスよく含まれているかも確認しましょう。
（特定の品種が少ないと、モデルがその品種を学習しにくくなります）

💡 ヒント：「Series の value_counts で各クラスの件数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# 分割後の行数を確認し、訓練データの品種ごとの件数バランスをチェックする

---
## 問1  モデルを学習させる ★☆☆

実務でのイメージ：
「過去の顧客データ（X_train, y_train）をもとに、新規顧客（X_test）の行動を予測するモデルを作る」

---

Q：RandomForestClassifier でモデルを作り、学習させてください

1. `model = RandomForestClassifier(...)` でモデルを作る
   📌 `(...)` には `n_estimators=100, random_state=42` を入れます。n_estimators はランダムに作る決定木の本数です（多いほど安定）。
   random_state=42 は乱数のシードを固定します（42 に特別な意味はありません）。誰が実行しても同じ結果になるよう再現性を確保するために指定します。
2. `model.fit(X_train, y_train)` で学習させる

💡 ヒント：「RandomForestClassifier でモデルを作って学習させる方法」を Copilot に聞いてみましょう。

In [ ]:
# RandomForest のモデルを作成して、学習用データで学習（fit）させる

---
問1-b（追加）  モデルの設定を確認しましょう

学習後に「何本の決定木を使ったか」「何個の特徴量を学習したか」を確認します。
📌 `n_features_in_` が **13** になっていれば正解です（品種名列を除いた 13 列で正しく学習できています）。
   もし 14 になっていたら → STEP 1 に戻って X から品種名列を除いてください。

💡 ヒント：「学習済み RandomForest モデルの木の本数・特徴量数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# モデルが何本の決定木を使い、何個の特徴量を学習したか確認する

---
問1-c（追加）  1件だけ取り出して予測してみましょう

fit()（勉強）が終わったら、predict()（試験）を実際に試してみましょう。
テストデータの最初の1件を取り出して、モデルが何品種と予測するかを確認します。

📌 スライドの「試験勉強（fit）→ 本番試験（predict）」が、たった1行で体験できます。

💡 ヒント：「DataFrame の特定の1行を取り出してモデルで予測する方法」を Copilot に聞いてみましょう。

In [ ]:
# テストデータの1件目を取り出して、モデルの予測と実際の品種を比べる

---
## 問2  テストデータで予測して正解率を確認する ★★☆

実務でのイメージ：
「新しい顧客データに対してモデルが正しく品種を当てられるか、正解率で評価する」

---

Q：テストデータで予測して、正解率を計算してください

1. `model.predict(X_test)` で予測する
2. `accuracy_score(y_test, y_pred)` で正解率を計算する

💡 ヒント：「学習済みモデルでテストデータを予測して正解率を計算する方法」を Copilot に聞いてみましょう。

In [ ]:
# テストデータに対してモデルで予測し、正解率を計算して評価する

---
問2-b（追加）  訓練精度と比べて「過学習していないか」確認しましょう

スライドで「訓練99%・テスト60%」の過学習の例を見ました。
今のモデルは訓練データでも予測させて、テスト精度と比べると過学習の有無が確認できます。

📌 差が小さい（3%以内） → 過学習なし：「本番試験でも勉強の成果が出た」
📌 差が大きい（10%以上）→ 過学習の可能性：「暗記しただけで理解していない」

💡 ヒント：「X_train に predict して accuracy_score で訓練精度を計算する方法」を Copilot に聞いてみましょう。

In [ ]:
# 訓練データでも予測して、訓練精度とテスト精度の差（過学習の程度）を確認する

---
### 問2 の気づき（AI 禁止：1 行でOK）

Q：正解率は何 % でしたか？目標の 90% を超えていましたか？

>

---
## 問3  「どこで間違えたか」と「何が重要か」を確認する ★★☆（最重要）

実務でのイメージ：
「どの品種（顧客セグメント）を間違えやすいか」を特定して改善する

---

Q3-1：混同行列を表示して、どの品種を間違えたか確認してください

混同行列は「実際の品種」×「予測した品種」の表です。
対角線以外のセルに数値があると「間違い」です。

💡 ヒント：「混同行列を計算して DataFrame で表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 混同行列を作成して、どの品種をどの品種と間違えたかを確認する

---
Q3-1（追加）  混同行列を「表形式」で見やすく表示しましょう

`confusion_matrix()` の出力は数値の配列で見づらいです。
品種名をラベルにした DataFrame に変換すると「どの品種をどの品種と間違えたか」が一目でわかります。

💡 ヒント：「confusion_matrix の結果を品種名付きの DataFrame に変換する方法」を Copilot に聞いてみましょう。

In [ ]:
# 混同行列を品種名付きの DataFrame に変換して、どのクラスを間違えたかを見やすく確認する

---
### Q3-1 の気づき（AI 禁止：1 行でOK）

Q：対角線以外のセルに数値はありましたか？あった場合、どの品種を間違えていましたか？

>

---
Q3-2：特徴量重要度を棒グラフで表示してください

RandomForest は「どの変数が品種の分類に役立ったか（重要度）」を自動的に計算します。
Ch.2 で立てた仮説（alcohol・proline が重要そう）は当たっていましたか？

💡 ヒント：「特徴量重要度を取得して高い順に棒グラフで表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 特徴量重要度と特徴量名をセットにした DataFrame を作る


In [ ]:
# 横棒グラフ用に重要度の小さい順に並べ替える（グラフは下から上に描かれるため大きいものが上になる）


In [ ]:
# 特徴量重要度を横棒グラフで表示する


---
Q3-2（追加）  重要度の数値を上位5件で確認しましょう

グラフで「どの変数が重要か」の傾向を確認したら、具体的な数値も確認しましょう。
1位と2位の差は大きいですか？Ch.2 の仮説（proline・alcohol が重要）は当たっていましたか？

💡 ヒント：「特徴量重要度を変数名とセットで上位5件を表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 特徴量重要度の DataFrame を作成して、重要度の大きい順に並べ替える


In [ ]:
# 上位5件を仮説と照合する


---
### Q3-2 の気づき（AI 禁止：1 行でOK）

Q：特徴量重要度 1 位の変数は何でしたか？Ch.2 の仮説は当たっていましたか？

>

---
問3 まとめ（追加）  全評価指標を一度に確認しましょう ★★☆

`classification_report` を使うと Precision・Recall・F1スコアを品種ごとに一覧で確認できます。
問4（発展）の準備として、まず全体像を俯瞰しておきましょう。

💡 ヒント：「classification_report で分類の詳細指標を一覧表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# Precision・Recall・F1スコアを品種ごとに一覧で確認する


---
## STEP 3  考察・振り返り （AI 禁止）

⛔ 注意 AI は使わないこと。自分の言葉で書いてください。

Q：Ch.2 で立てた仮説（重要な変数の予想）は当たっていましたか？

>

Q：正解率が約 97% というのは「良いモデル」だと思いますか？その理由は？

💡 参考：実務では 70〜80% でも「使えるモデル」として採用されることがあります。何 % 以上が合格かはビジネスの文脈によります。

>

In [ ]:
# 考察を記入したら、問4（発展）へ進んでください（時間が余った人のみ）

---
## 問4（発展）  予測の詳細を確認する ★★★

📋 発展 時間が余った人だけ取り組んでください

実務でのイメージ：

📌 **Precision（適合率）**：「wine_X と予測した中で、本当に wine_X だった割合」
　例）「陽性と診断した患者のうち、本当に病気だった割合」

📌 **Recall（再現率）**：「本当に wine_X のうち、正しく wine_X と予測できた割合」
　例）「本当に病気の患者のうち、正しく陽性と診断できた割合」

→ 「見逃したくない（病気なのに陰性と言いたくない）」→ **Recall を高める**
→ 「誤診をなくしたい（病気でないのに陽性と言いたくない）」→ **Precision を高める**

---

Q：品種ごとの Precision・Recall を計算して表示してください

💡 ヒント：「クラスごとの Precision と Recall を計算する方法」を Copilot に聞いてみましょう。

In [ ]:
# 品種ごとの Precision（適合率）と Recall（再現率）を計算して、より詳しい評価指標を確認する

---
## お疲れさまでした！

| 操作 | 発見 | ビジネスへの接続 |
|------|------|----------------|
| モデル学習 | 13 変数で自動的にルールを作成 | 人間のルール作りを自動化 |
| 正解率 97% | ほぼすべての品種を正しく分類 | 現場で使えるモデルの基準 |
| 混同行列 | wine_2 の一部が wine_1 に誤分類 | 改善が必要な品種がわかる |
| 特徴量重要度 | proline が最重要 → Ch.2 の仮説と一致 | EDA → ML の流れが完結 |